# บทที่ 5: Queue และ Deque (คิว)> **รายวิชา:** Data Structures and Algorithms (สำหรับนิสิต IT ชั้นปีที่ 2)> **อ้างอิงแนวเนื้อหา:** Narasimha Karumanchi, *Data Structures and Algorithmic Thinking with Python*> **วิธีใช้:** อ่านคำอธิบาย → รันโค้ดทีละเซลล์ → ลองแก้ค่า input → ทำแบบฝึกหัดท้ายบท (เฉลยอยู่ท้ายไฟล์)---

## 5.1 แนวคิด FIFOQueue ทำงานแบบ **FIFO** (First In, First Out) เข้าทาง *rear* ออกทาง *front*| Operation | Complexity ||---|---|| `enqueue(x)` | \(O(1)\) || `dequeue()` | \(O(1)\) || `front()` | \(O(1)\) |**ข้อควรระวัง:** ถ้าใช้ `list.pop(0)` จะเป็น \(O(n)\) เพราะต้องเลื่อนสมาชิกทั้งหมด → ใช้ **circular array** หรือ `collections.deque`**การใช้งานจริง:** งานพิมพ์ (print spooler), BFS, scheduler ของ OS, message queue, buffer สตรีมข้อมูล

In [ ]:
# แบบที่ 1: Circular Queue บนอาร์เรย์ขนาดคงที่ -> enqueue/dequeue = O(1)class CircularQueue:    def __init__(self, cap):        self.a = [None] * cap        self.cap, self.front, self.n = cap, 0, 0    def is_empty(self): return self.n == 0    def is_full(self):  return self.n == self.cap    def enqueue(self, x):        if self.is_full(): raise OverflowError('queue is full')        rear = (self.front + self.n) % self.cap     # หมุนกลับต้นอาร์เรย์        self.a[rear] = x        self.n += 1    def dequeue(self):        if self.is_empty(): raise IndexError('queue is empty')        v = self.a[self.front]        self.a[self.front] = None        self.front = (self.front + 1) % self.cap        self.n -= 1        return v    def peek(self): return self.a[self.front]    def __len__(self): return self.n    def __repr__(self):        items = [self.a[(self.front + i) % self.cap] for i in range(self.n)]        return f"Queue(front->rear: {items}) raw={self.a}"q = CircularQueue(5)for x in (1, 2, 3, 4):    q.enqueue(x)print(q)print("dequeue:", q.dequeue(), q.dequeue())q.enqueue(5); q.enqueue(6)          # สังเกตว่า index หมุนกลับไปต้นอาร์เรย์print(q)

In [ ]:
# แบบที่ 2: Queue จาก linked list (เก็บทั้ง front และ rear) -> O(1) ทั้งสองฝั่ง ไม่จำกัดขนาดclass LinkedQueue:    class _N:        __slots__ = ('v', 'nxt')        def __init__(self, v): self.v, self.nxt = v, None    def __init__(self):        self.front = self.rear = None        self.n = 0    def enqueue(self, x):        node = self._N(x)        if self.rear: self.rear.nxt = node        else:         self.front = node        self.rear = node        self.n += 1    def dequeue(self):        if not self.front: raise IndexError('empty')        v = self.front.v        self.front = self.front.nxt        if not self.front: self.rear = None        self.n -= 1        return v    def __len__(self): return self.nlq = LinkedQueue()for c in "XYZ": lq.enqueue(c)print([lq.dequeue() for _ in range(3)])     # -> X Y Z (FIFO)

In [ ]:
# แบบที่ 3 (แนะนำใช้งานจริง): collections.deque = double-ended queue, O(1) ทั้งสองปลายfrom collections import dequedq = deque([1, 2, 3])dq.append(4)        # ต่อท้ายdq.appendleft(0)    # ต่อหัวprint(dq)print("pop:", dq.pop(), "| popleft:", dq.popleft(), "->", dq)# deque ที่จำกัดขนาด เหมาะทำ sliding window / log ล่าสุด N รายการrecent = deque(maxlen=3)for x in range(1, 7):    recent.append(x)print("3 ค่าล่าสุด:", recent)

In [ ]:
# Queue จาก 2 stacks (โจทย์คลาสสิก) - amortized O(1) ต่อ operationclass QueueFromStacks:    def __init__(self):        self.inbox, self.outbox = [], []    def enqueue(self, x):        self.inbox.append(x)    def dequeue(self):        if not self.outbox:                    # ย้ายทั้งก้อนเมื่อ outbox ว่าง            while self.inbox:                self.outbox.append(self.inbox.pop())        if not self.outbox: raise IndexError('empty')        return self.outbox.pop()qs = QueueFromStacks()for x in (1, 2, 3): qs.enqueue(x)print(qs.dequeue(), qs.dequeue())qs.enqueue(4)print(qs.dequeue(), qs.dequeue())

In [ ]:
# ประยุกต์: sliding window maximum ด้วย monotonic deque -> O(n)from collections import dequedef window_max(a, k):    dq, out = deque(), []                      # เก็บ index เรียงค่าจากมากไปน้อย    for i, x in enumerate(a):        while dq and a[dq[-1]] <= x:            dq.pop()                           # ตัวที่เล็กกว่าไม่มีโอกาสเป็น max        dq.append(i)        if dq[0] <= i - k:            dq.popleft()                       # ตัวที่หลุดหน้าต่าง        if i >= k - 1:            out.append(a[dq[0]])    return outprint(window_max([1, 3, -1, -3, 5, 3, 6, 7], 3))

---## แบบฝึกหัดบทที่ 51. สร้าง stack จาก queue 2 อัน (กลับกับโจทย์ในบท)2. สลับสมาชิก k ตัวแรกของคิว โดยลำดับที่เหลือคงเดิม3. ใช้คิวสร้างเลขฐานสองของ 1..n ตามลำดับ (`1, 10, 11, 100, ...`)4. คำนวณ Hot Potato / Josephus problem ด้วยคิวหมุน

### เฉลยบทที่ 5

In [ ]:
from collections import deque# ข้อ 1: ทำให้ push แพง (O(n)) แต่ pop ถูก (O(1))class StackFromQueues:    def __init__(self):        self.q = deque()    def push(self, x):        self.q.append(x)        for _ in range(len(self.q) - 1):            self.q.append(self.q.popleft())    # หมุนให้ตัวใหม่มาอยู่หน้าสุด    def pop(self):        return self.q.popleft()# ข้อ 2: ใช้ stack กลับลำดับ k ตัวแรก แล้วหมุนส่วนที่เหลือdef reverse_first_k(q, k):    st = [q.popleft() for _ in range(k)]    while st:        q.append(st.pop())    for _ in range(len(q) - k):        q.append(q.popleft())    return q# ข้อ 3: BFS บนต้นไม้ของสตริงเลขฐานสองdef binary_numbers(n):    out, q = [], deque(['1'])    for _ in range(n):        s = q.popleft()        out.append(s)        q.append(s + '0'); q.append(s + '1')    return out# ข้อ 4: Josephus - ส่งมือถัดไป k-1 ครั้ง แล้วคัดคนที่ k ออกdef josephus(names, k):    q = deque(names)    order = []    while q:        for _ in range(k - 1):            q.append(q.popleft())        order.append(q.popleft())    return orders = StackFromQueues()for x in (1, 2, 3): s.push(x)print("stack pop:", s.pop(), s.pop())print(reverse_first_k(deque([1,2,3,4,5]), 3))print(binary_numbers(7))print(josephus(list("ABCDE"), 3))